In [ ]:
import pandas as pd
from sklearn import set_config

df = pd.read_csv("../data/processed/paper_01.csv")
drop_cols = ["Text_Title", "Cutoff_Year", "Cutoff_Session", "Yrs_Since_Last"]
X = df.drop(columns=["App_Next"])
y = df[["App_Next"]]
groups = X.groupby(['Cutoff_Year', 'Cutoff_Session', 'Paper_Stream']).size().values

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

set_config(transform_output="pandas")
print(f"Numeric columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")

KeyboardInterrupt: 

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import re

REDUCE_OPS = {
    "add": lambda x: x.sum(axis=1),
    "mul": lambda x: np.prod(x, axis=1),
    "mean": lambda x: x.mean(axis=1),
    "max": lambda x: x.max(axis=1),
    "min": lambda x: x.min(axis=1),
}

INTERACT_OPS = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
    "div": lambda a, b: a / (b + 1e-6),
}

class InteractionAdder(BaseEstimator, TransformerMixin):
    def __init__(self, interactions):
        self.interactions = interactions

    def fit(self, X, y=None):
        self.feature_names_in_ = list(X.columns)
        self._col_map = {orig: self._sanitize(orig) for orig in self.feature_names_in_}
        self.feature_names_out_ = list(self._col_map.values())
        return self

    def _sanitize(self, name):
        return re.sub(r"[^A-Za-z0-9_]", "_", name)

    def make_name(self, cols, reduce_op, interact_col, interact_op):
        if reduce_op is None:
            base = cols[0]
        else:
            base = f"{reduce_op}(" + ",".join(cols) + ")"
        if interact_col is not None:
            name = f"{base}_{interact_op}_{interact_col}"
        else:
            name = base
        return self._sanitize(name)

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_

    def transform(self, X):
        X = X.copy()
        X.rename(columns=self._col_map, inplace=True)

        for cols, reduce_op, interact_col, interact_op, feat_name in self.interactions:
            sanitized_cols = [self._col_map.get(c, self._sanitize(c)) for c in cols]
            base = X[sanitized_cols]

            if reduce_op is None:
                if base.shape[1] != 1:
                    raise ValueError("reduce_op is None but multiple cols were provided")
                feat = base.iloc[:, 0].to_numpy()
            else:
                feat = REDUCE_OPS[reduce_op](base.values)

            if interact_col is not None:
                interact_col_safe = self._col_map.get(interact_col, self._sanitize(interact_col))
                feat = INTERACT_OPS[interact_op](feat, X[interact_col_safe].values)

            if feat_name is not None:
                name = self._sanitize(feat_name)
            else:
                name = self.make_name(cols, reduce_op, interact_col, interact_op)

            X[name] = feat
            self.feature_names_out_.append(name)

        return X


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMRanker
from sklearn.compose import ColumnTransformer

def check_ranking_data(X, y, groups):
    y_arr = np.asarray(y).ravel()
    print("rows:", len(y_arr))
    print("sum(groups):", np.sum(groups), "rows match:", np.sum(groups) == len(y_arr))
    print("overall unique labels:", np.unique(y_arr).size)
    # build group indices from `groups` sizes
    group_idx = np.repeat(np.arange(len(groups)), groups)
    if len(group_idx) != len(y_arr):
        print("ERROR: group sizes don't sum to number of rows")
        return
    per_group_nunique = pd.Series(y_arr).groupby(group_idx).nunique()
    print("per-group label nunique distribution:")
    print(per_group_nunique.value_counts().sort_index())
    # show how many groups have only one label
    print("groups with single unique label:", (per_group_nunique == 1).sum())

interactions = [
    (["Appd_Last_1" , "Appd_Last_2"], "add", "Appd_Last_3", "add", "Recent_Frequency"),
    (["Num_Stream_App"], None, "App_Total", "div", "Stream_App_Ratio"),
    (["Sessions_Since_Last", "App_Total"], "mul", None, None, None)
]

num_transformer = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("interaction_adder", InteractionAdder(interactions))
])

cat_transformer = Pipeline([
    ("onehot", OneHotEncoder(sparse_output=False, handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols),
], remainder="drop")

ranker = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    boosting_type="gbdt",
    n_estimators=100,
    min_child_samples=1,
    min_data_in_leaf=1,
    min_split_gain=0.0,
    learning_rate=0.1
)

full_pipeline = Pipeline([
    ("pre", preprocessor),
    ("ranker", ranker),
])

y_arr = np.asarray(y).ravel()
group_idx = np.repeat(np.arange(len(groups)), groups)
per_group_nunique = pd.Series(y_arr).groupby(group_idx).nunique()
single_label_groups = per_group_nunique[per_group_nunique == 1].index.tolist()

print(f"Removing {len(single_label_groups)} with unique label: {single_label_groups}")

mask = ~np.isin(group_idx, single_label_groups)
X_filtered = X[mask].reset_index(drop=True)
y_filtered = y_arr[mask]

print(f"X Filtered: {X_filtered}")
groups_filtered = X_filtered.groupby(['Cutoff_Year', 'Cutoff_Session', 'Paper_Stream']).size().values
X = df.drop(columns=drop_cols)

print(f"Filtered: {len(X_filtered)} rows (from {len(X)}), {len(groups_filtered)} groups (from {len(groups)})")

# verify the fix
check_ranking_data(X_filtered, y_filtered, groups_filtered)

# retrain with filtered data
full_pipeline.fit(X_filtered, y_filtered, ranker__group=groups_filtered)

Removing 11 with unique label: [0, 2, 5, 9, 10, 21, 23, 25, 27, 29, 31]
X Filtered:                                      Text_Title  Cutoff_Year Cutoff_Session  \
0                           A Passage to Africa         2018       May/June   
1                           A Passage to Africa         2018       May/June   
2                                 H is for Hawk         2018       May/June   
3                                 H is for Hawk         2018       May/June   
4                            Chinese Cinderella         2018       May/June   
..                                          ...          ...            ...   
205                The Danger of a Single Story         2025        Oct/Nov   
206         A Game of Polo with a Headless Goat         2025        Oct/Nov   
207         A Game of Polo with a Headless Goat         2025        Oct/Nov   
208  127 Hours: Between a Rock and a Hard Place         2025        Oct/Nov   
209  127 Hours: Between a Rock and a Hard Place

LightGBMError: Do not support special JSON characters in feature name.